In [1]:
import nibabel as nib
import os
import matplotlib.pyplot as plt
import numpy as np
from dipy.io.gradients import read_bvals_bvecs
from fury import window, actor

# Data and Class imports
# from data import data, affine, labels, bvals, bvecs, vox_size
from env_main_updated import UnifiedTractographyEnv
from agent import BranchingStreamlineAgent,UnifiedBranchingAgent
from dipy.viz import colormap
from dipy.tracking import utils

In [4]:
anat_img=nib.load(r"../Deepa/data.nii.gz")
affine=anat_img.affine
data=anat_img.get_fdata()
vals=anat_img.header.get_zooms()
vox_list = [float(v) for v in vals]
vox_size=vox_list[:3]
vox_size

[1.25, 1.25, 1.25]

In [5]:
bvals_path="../Deepa/bvals"
bvals=np.loadtxt(bvals_path)
bvecs_path="../Deepa/bvecs"
bvecs=np.loadtxt(bvecs_path)
print(bvals.shape,bvecs.shape)

(288,) (3, 288)


In [6]:
bvecs=bvecs.T

In [7]:
mask_path="../Deepa/nodif_brain_mask.nii.gz"
mask=nib.load(mask_path).get_fdata()

In [8]:
mask.shape

(145, 174, 145)

In [9]:
coords=np.argwhere((mask==True) )
print(coords.shape)

(730310, 3)


In [10]:
from fury import window,actor

In [11]:
scene = window.Scene()
points=np.array(coords)
point_actor=actor.point(points,colors=(1,0,0),point_radius=0.25)
scene.add(point_actor)
window.show(scene,size=(800,800))

In [59]:
mid=mask.shape[2]//2
seeds=mask[:,:,mid-20:mid+5]
seed_coords=np.argwhere(seeds)
print(seed_coords.shape)

(274293, 3)


In [ ]:
scene = window.Scene()
points=np.array(seed_coords)
point_actor=actor.point(points,colors=(1,0,0),point_radius=0.25)
scene.add(point_actor)
window.show(scene,size=(800,800))

In [23]:
n_points = 1000  # how many you want to keep
idx = np.random.choice(seed_coords.shape[0], n_points, replace=False)
reduced_coords = seed_coords[idx]

print(reduced_coords.shape)


(1000, 3)


In [25]:
scene = window.Scene()
points=np.array(reduced_coords)
point_actor=actor.point(points,colors=(1,0,0),point_radius=0.25)
scene.add(point_actor)
window.show(scene,size=(800,800))

In [34]:
import nibabel as nib
import numpy as np


# 2. Initialize the labels array with zeros (Background)
labels = np.zeros(mask.shape, dtype=np.int32)

# 3. Assign Tissue Types
labels[mask > 0.5] = 1  # CSF


# 4. Refine Seeds (Subset of White Matter)
# If you want to seed only from the center of the brain (e.g., Corpus Callosum)
# we can mask out everything except a central box in the WM
z_mid = mask.shape[2] // 2
seed_mask = np.zeros_like(mask)
# Create a slab 10 voxels thick in the middle
seed_mask[:, :, z_mid-10 : z_mid+10] = mask[:,:,z_mid-10:z_mid+10]

# Update labels: Only WM voxels in that central slab become '2' (Seeds)
# The rest of the WM stays as a different label (e.g., 4) or remains 2
final_labels = labels.copy()
final_labels[ (seed_mask == 1)] = 2 # Actual Seeds

# 5. Save the new label map
new_label_img = nib.Nifti1Image(final_labels, affine)
nib.save(new_label_img, 'data_processed_labels.nii.gz')

In [35]:
preprocessed_labels=nib.load('data_processed_labels.nii.gz')
labels_data=preprocessed_labels.get_fdata()
labels_data.shape

(145, 174, 145)

In [39]:
coords=np.argwhere(labels_data==2)
coords.shape

(209928, 3)

In [40]:
n_samples=1000
indices = np.random.choice(coords.shape[0], n_samples, replace=False)
reduced_matrix = coords[indices]
reduced_matrix.shape


(1000, 3)

In [41]:
scene = window.Scene()
points=np.array(coords)
point_actor=actor.point(points,colors=(1,0,0),point_radius=0.25)
scene.add(point_actor)
window.show(scene,size=(800,800))

In [45]:
env_data = UnifiedTractographyEnv(
    data=data, affine=affine, labels=labels_data, 
    bvals=bvals, bvecs=bvecs, vox_size=vox_size,
    fa_threshold=0.25, max_curvature_deg=45
)

In [42]:
anat_img=nib.load(r"../Deepa/cnn_k9_fwe_dwi.nii.gz")
affine=anat_img.affine
data_cnn=anat_img.get_fdata()
vals=anat_img.header.get_zooms()
vox_list = [float(v) for v in vals]
vox_size=vox_list[:3]
vox_size

[1.25, 1.25, 1.25]

In [44]:
anat_img=nib.load(r"../Deepa/vit_k9_fwe_dwi.nii.gz")
affine=anat_img.affine
data_vit=anat_img.get_fdata()
vals=anat_img.header.get_zooms()
vox_list = [float(v) for v in vals]
vox_size=vox_list[:3]
vox_size

[1.25, 1.25, 1.25]

In [ ]:
env_cnn = UnifiedTractographyEnv(
    data=data_cnn, affine=affine, labels=labels_data, 
    bvals=bvals, bvecs=bvecs, vox_size=vox_size,
    fa_threshold=0.25, max_curvature_deg=45
)

In [ ]:
env_vit = UnifiedTractographyEnv(
    data=data_vit, affine=affine, labels=labels_data, 
    bvals=bvals, bvecs=bvecs, vox_size=vox_size,
    fa_threshold=0.25, max_curvature_deg=45
)

In [46]:
agent = UnifiedBranchingAgent(env_data)

In [51]:
seed_indices = utils.seeds_from_mask((labels_data == 2),affine=np.eye(4),density=(1,1,1))
print(seed_indices.shape)

(209928, 3)


In [53]:
n_samples=2500
indices = np.random.choice(seed_indices.shape[0], n_samples, replace=False)
reduced_matrix = seed_indices[indices]
reduced_matrix.shape


(2500, 3)

In [54]:
seed_indices=reduced_matrix
#seed_indices = utils.seeds_from_mask((labels_data == 2),affine=np.eye(4),density=(1,1,1))
np.random.shuffle(seed_indices)
num_seeds =  len(seed_indices)
scene = window.Scene()
all_streamlines = []

print(f"Starting branching tracking for {num_seeds} seeds...")

for i in range(num_seeds):
    seed = seed_indices[i].astype(np.float32)
    
    # This returns a LIST of streamlines because of branching
    branches = agent.track(seed)
    
    if branches:
        all_streamlines.extend(branches)
        print(f"Seed {i+1}: Found {len(branches)} branches.")

# 4. Rendering in Fury
if all_streamlines:
    print(f"Total streamlines generated: {len(all_streamlines)}")
    
    # Use coloring based on local orientation for visual clarity
    env_data.render_streamlines(scene,all_streamlines)
    env_data.render_wm_surface(scene)
    
    # Add the white matter surface at low opacity for anatomical context
    # env.render_wm_surface(scene, opacity=0.05)
 
    scene.reset_camera()
    
    print("Opening interactive window...")
    window.show(scene, size=(1024, 768), title="Branching Fiber Tracking")
else:
    print("No streamlines found. Check peak thresholds.")

Starting branching tracking for 2500 seeds...
Seed 3: Found 1 branches.
Seed 4: Found 1 branches.
Seed 6: Found 67 branches.
Seed 7: Found 1 branches.
Seed 8: Found 1 branches.
Seed 9: Found 1 branches.
Seed 10: Found 9 branches.
Seed 11: Found 1 branches.
Seed 12: Found 1 branches.
Seed 13: Found 1 branches.
Seed 14: Found 9 branches.
Seed 17: Found 14 branches.
Seed 20: Found 1 branches.
Seed 21: Found 1 branches.
Seed 23: Found 1 branches.
Seed 24: Found 4 branches.
Seed 28: Found 1 branches.
Seed 29: Found 2 branches.
Seed 30: Found 38 branches.
Seed 31: Found 8 branches.
Seed 32: Found 1 branches.
Seed 33: Found 1 branches.
Seed 34: Found 1 branches.
Seed 35: Found 1 branches.
Seed 36: Found 2 branches.
Seed 39: Found 1 branches.
Seed 41: Found 1 branches.
Seed 44: Found 42 branches.
Seed 45: Found 1 branches.
Seed 51: Found 63 branches.
Seed 52: Found 1 branches.
Seed 54: Found 1 branches.
Seed 55: Found 4 branches.
Seed 56: Found 1 branches.
Seed 58: Found 1 branches.
Seed 59: F

In [56]:
from dipy.io.streamline import save_trk
from dipy.io.stateful_tractogram import Space, StatefulTractogram
sft = StatefulTractogram(all_streamlines, anat_img, Space.RASMM)
save_trk(sft, "trk/data_2500.trk", all_streamlines)

In [57]:
from nibabel.streamlines import load as load_trk
trk=load_trk( "trk/data_2500.trk")
streamlines=trk.streamlines

In [58]:
scene = window.Scene()
color = colormap.line_colors(streamlines)
scene.add(actor.line(streamlines, color))
# Save a snapshot
window.record(scene=scene, out_path='data_2500.png', size=(900, 900))
# Interactive display (works if display available)
window.show(scene)

C:\Users\Samsung\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\IPython\core\interactiveshell.py:3508: UserWarning: We'll no longer accept the way you call the line function in future versions of FURY.

Here's how to call the Function line: line(lines_value, colors='value', opacity='value', linewidth='value', spline_subdiv='value', lod='value', lod_points='value', lod_points_size='value', lookup_colormap='value', depth_cue='value', fake_tube='value')

  exec(code_obj, self.user_global_ns, self.user_ns)


In [6]:
from nibabel.streamlines import load as load_trk
trk=load_trk( "trk/deepa/vit_10000.trk")
streamlines=trk.streamlines

In [2]:
from dipy.viz import window, actor, colormap

In [7]:
scene = window.Scene()
color = colormap.line_colors(streamlines)
scene.add(actor.line(streamlines, color))
# Save a snapshot
window.record(scene=scene, out_path='vit_10000.png', size=(900, 900))
# Interactive display (works if display available)
window.show(scene)

In [8]:
type(streamlines)

nibabel.streamlines.array_sequence.ArraySequence

In [11]:
import numpy as np
from nibabel.streamlines import ArraySequence
def streamline_length(sl):
    # Euclidean distances between consecutive points
    diffs = np.diff(sl, axis=0)
    seg_lengths = np.sqrt((diffs ** 2).sum(axis=1))
    return seg_lengths.sum()

# Set your minimum length threshold
min_length = 30.0  # in mm, adjust as needed

# Filter streamlines
filtered_streamlines = ArraySequence([sl for sl in streamlines if streamline_length(sl) >= min_length])


In [12]:
scene = window.Scene()
color = colormap.line_colors(filtered_streamlines)
scene.add(actor.line(filtered_streamlines, color))
# Save a snapshot
window.record(scene=scene, out_path='vit_10000_filtered.png', size=(900, 900))
# Interactive display (works if display available)
window.show(scene)

C:\Users\Samsung\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\IPython\core\interactiveshell.py:3508: UserWarning: We'll no longer accept the way you call the line function in future versions of FURY.

Here's how to call the Function line: line(lines_value, colors='value', opacity='value', linewidth='value', spline_subdiv='value', lod='value', lod_points='value', lod_points_size='value', lookup_colormap='value', depth_cue='value', fake_tube='value')

  exec(code_obj, self.user_global_ns, self.user_ns)
